# Agentic Pattern - Reflection
The Reflection pattern involves an agent evaluating its own work, output, or internal state and using that evaluation to improve its performance or refine its response.

[![Agentic Pattern - Reflection presentation](https://img.youtube.com/vi/Zbmhua5ovEc/0.jpg)](https://youtu.be/Zbmhua5ovEc) 
<br />
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
<a href="https://www.youtube.com/@dayonedev" target="_new">
  <img align="center" src="https://img.shields.io/youtube/channel/views/UCiLziPE9aPxCouSsX0lJ--A" />
</a>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
<a href="https://linkedin.com/in/krishnamanchikalapudi" target="_new">
  <img align="center" src="https://img.shields.io/badge/linkedin-%230077B5.svg?style=for-the-badge&logo=linkedin&logoColor=white" />
</a>
<br/>

In [1]:
%%time
import warnings
warnings.filterwarnings("ignore")

!python3 -m pip install --upgrade pip

CPU times: user 9.48 ms, sys: 14.7 ms, total: 24.2 ms
Wall time: 1.05 s


In [2]:
%pip install -U -q langchain langchain-ollama ipython-autotime --use-deprecated=legacy-resolver

%load_ext autotime

Note: you may need to restart the kernel to use updated packages.
time: 112 μs (started: 2025-09-18 22:10:43 -07:00)


## Variables

In [3]:
my_model_ollama = "llama3.2"

time: 348 μs (started: 2025-09-18 22:10:43 -07:00)


## Initialize OLLAM service
OLLAMA inference client is initialized to interact with the OLLAMA API for generating responses from the specified model.

In [4]:
from langchain_ollama.llms import OllamaLLM

llm_client = OllamaLLM(
    model=my_model_ollama,
    base_url="http://localhost:11434",
    headers={"Content-Type": "application/json"},
    stream=True,
    temperature=0.7,
)

llm_client

OllamaLLM(model='llama3.2', temperature=0.7, base_url='http://localhost:11434')

time: 1.61 s (started: 2025-09-18 22:10:43 -07:00)


### TEST llm_client with a simple prompt

In [5]:
response = llm_client.invoke("What is Agentic Pattern - Reflection")

print(f"{response}")

Agentic pattern of reflection refers to a specific approach in learning and personal development that involves taking an active, deliberate, and intentional stance towards one's own learning process. It emphasizes the individual's agency and autonomy in shaping their own knowledge, skills, and experiences.

In this context, "reflection" refers to the act of intentionally examining and evaluating one's own thoughts, feelings, and actions, with the goal of gaining new insights, making connections between ideas, and developing a deeper understanding of oneself and the world.

Agentic pattern of reflection involves several key characteristics:

1. **Self-directedness**: The individual takes charge of their own learning process, setting goals and making decisions about what they want to learn and how they want to approach it.
2. **Intentionality**: The person reflects on their thoughts, feelings, and actions with a specific purpose in mind, such as gaining new insights or developing new ski

## Define Chain Components

1. Initial Generation: Creates the first draft of the product description. The input to this chain will be a dictionary, so we update the prompt template.

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

generation_chain = (
    ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "Write a short, simple product description for a new smart coffee mug.",
            ),
            ("user", "{product_details}"),
        ]
    )
    | llm_client
    | StrOutputParser()
)

generation_chain

ChatPromptTemplate(input_variables=['product_details'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Write a short, simple product description for a new smart coffee mug.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['product_details'], input_types={}, partial_variables={}, template='{product_details}'), additional_kwargs={})])
| OllamaLLM(model='llama3.2', temperature=0.7, base_url='http://localhost:11434')
| StrOutputParser()

time: 25.5 ms (started: 2025-09-18 22:10:52 -07:00)


2. Critique: Evaluates the generated description and provides feedback.

In [7]:
critique_chain = (
    ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """Critique the following product description based on clarity, conciseness, and appeal.
        Provide specific suggestions for improvement.""",
            ),
            # This will receive 'initial_description' from the previous step.
            ("user", "Product Description to Critique:\n{initial_description}"),
        ]
    )
    | llm_client
    | StrOutputParser()
)

critique_chain

ChatPromptTemplate(input_variables=['initial_description'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Critique the following product description based on clarity, conciseness, and appeal.\n        Provide specific suggestions for improvement.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['initial_description'], input_types={}, partial_variables={}, template='Product Description to Critique:\n{initial_description}'), additional_kwargs={})])
| OllamaLLM(model='llama3.2', temperature=0.7, base_url='http://localhost:11434')
| StrOutputParser()

time: 1.75 ms (started: 2025-09-18 22:10:52 -07:00)


3. Refinement: Rewrites the description based on the original details and the critique.

In [8]:
refinement_chain = (
    ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """Based on the original product details and the following critique,
        rewrite the product description to be more effective.

        Original Product Details: {product_details}
        Critique: {critique}

        Refined Product Description:""",
            ),
            (
                "user",
                "",
            ),  # User input is empty as the context is provided in the system message
        ]
    )
    | llm_client
    | StrOutputParser()
)

refinement_chain

ChatPromptTemplate(input_variables=['critique', 'product_details'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['critique', 'product_details'], input_types={}, partial_variables={}, template='Based on the original product details and the following critique,\n        rewrite the product description to be more effective.\n\n        Original Product Details: {product_details}\n        Critique: {critique}\n\n        Refined Product Description:'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template=''), additional_kwargs={})])
| OllamaLLM(model='llama3.2', temperature=0.7, base_url='http://localhost:11434')
| StrOutputParser()

time: 1.55 ms (started: 2025-09-18 22:10:52 -07:00)


## Build the Full Reflection Chain
This chain is structured to be more readable and linear.

In [9]:
from langchain_core.runnables import RunnablePassthrough

full_reflection_chain = (
    RunnablePassthrough.assign(initial_description=generation_chain)
    | RunnablePassthrough.assign(critique=critique_chain)
    | refinement_chain
)

full_reflection_chain

RunnableAssign(mapper={
  initial_description: ChatPromptTemplate(input_variables=['product_details'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Write a short, simple product description for a new smart coffee mug.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['product_details'], input_types={}, partial_variables={}, template='{product_details}'), additional_kwargs={})])
                       | OllamaLLM(model='llama3.2', temperature=0.7, base_url='http://localhost:11434')
                       | StrOutputParser()
})
| RunnableAssign(mapper={
    critique: ChatPromptTemplate(input_variables=['initial_description'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Critique the following product descript

time: 11.4 ms (started: 2025-09-18 22:10:52 -07:00)


## Run the Chain

In [10]:
product_details = (
    "A mug that keeps coffee hot and can be controlled by a smartphone app."
)

final_refined_description = full_reflection_chain.invoke(
    {"product_details": product_details}
)

print(f"{final_refined_description}")

Here is a refined product description incorporating the suggested improvements:

**Introducing the Smart Mug - Your Perfect Cup, Every Time**

Stay one step ahead of lukewarm coffee with our revolutionary Smart Mug. Say goodbye to bland, uninspiring brews and hello to a consistently perfect cup.

**Perfectly Brewed, Every Time**
Schedule your coffee brewing with ease using our intuitive smartphone app. Monitor the temperature of your mug and get reminders when it's time to refill - so you never miss out on that first sip.

**Experience the Ultimate Coffee Experience**

Enjoy perfectly brewed coffee every time
Stay in control of your coffee experience
Get the most out of your favorite brew

**Order Now and Start Sipping on Perfection**
Don't settle for anything less. Order your Smart Mug today and start enjoying a perfect cup of coffee, every time.

I incorporated the following changes:

1. Broke up long sentences into shorter, punchier ones to improve readability.
2. Used simpler langu

<br />
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
Subscribe to <a href="https://www.youtube.com/@dayonedev" target="_new">
  <img align="center" src="https://img.shields.io/youtube/channel/views/UCiLziPE9aPxCouSsX0lJ--A" />
</a>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
Connect me at <a href="https://linkedin.com/in/krishnamanchikalapudi" target="_new">
  <img align="center" src="https://img.shields.io/badge/linkedin-%230077B5.svg?style=for-the-badge&logo=linkedin&logoColor=white" />
</a>
<br/>